In [14]:
def get_puzzle_state(prompt_message):
    while True:
        print(prompt_message)
        print("enter 9 numbers (0-8) separated by spaces. Example: '1 2 3 4 5 6 7 8 0'")

        user_input = input("> ").strip().lower()

        if user_input in ['d', 'default']:
            # A common solvable 8-puzzle initial state that requires moves
            # For goal state, this will represent the standard solved state
            return [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
        try:
            numbers = list(map(int, user_input.split()))
            if len(numbers) != 9:
                print("Error: Please enter exactly 9 numbers or 'default'.")
                continue
            if not all(0 <= n <= 8 for n in numbers):
                print("Error: Numbers must be between 0 and 8.")
                continue
            if len(set(numbers)) != 9:
                print("Error: Numbers must be unique and represent all numbers from 0 to 8.")
                continue
            state = [numbers[i:i+3] for i in range(0, 9, 3)]
            return state
        except ValueError:
            print("Error: Invalid input. Please enter numbers separated by spaces or 'default'.")

# Get initial state from the user
initial_state = get_puzzle_state("Enter the initial state of the 8-puzzle:")

# Get goal state from the user
goal_state = get_puzzle_state("Enter the goal state of the 8-puzzle:")

Enter the initial state of the 8-puzzle:
enter 9 numbers (0-8) separated by spaces. Example: '1 2 3 4 5 6 7 8 0'
> 2 8 3 1 6 4 7 0 5
Enter the goal state of the 8-puzzle:
enter 9 numbers (0-8) separated by spaces. Example: '1 2 3 4 5 6 7 8 0'
> 1 2 3 8 0 4 7 6 5


In [16]:
from collections import deque

def dfs_solve_8_puzzle(initial_board_tuple, goal_board_tuple):
    queue = deque([(initial_board_tuple, [])])  # (board_tuple, path_list)
    visited = {initial_board_tuple}

    # Function to find the position of the blank (0) tile
    def find_blank(board):
        for r in range(3):
            for c in range(3):
                if board[r][c] == 0:
                    return r, c
        return -1, -1

    # Function to get possible moves from the current state
    def get_moves(board_tuple):
        board = [list(row) for row in board_tuple]  # Convert tuple back to list for modification
        r, c = find_blank(board)
        possible_moves = []

        # Possible directions: (dr, dc, move_name)
        directions = {
            'Up': (-1, 0),
            'Down': (1, 0),
            'Left': (0, -1),
            'Right': (0, 1)
        }

        for move_name, (dr, dc) in directions.items():
            new_r, new_c = r + dr, c + dc

            if 0 <= new_r < 3 and 0 <= new_c < 3:
                # Make the move: swap blank with the tile in the new position
                new_board = [row[:] for row in board]  # Deep copy
                new_board[r][c], new_board[new_r][new_c] = new_board[new_r][new_c], new_board[r][c]
                possible_moves.append((tuple(tuple(row) for row in new_board), move_name))
        return possible_moves

    while queue:
        current_board, current_path = queue.pop()  # Use pop() for DFS-like behavior (last in, first out)

        if current_board == goal_board_tuple:
            return len(current_path), current_path

        # Generate next moves
        for next_board, move in get_moves(current_board):
            if next_board not in visited:
                visited.add(next_board)
                queue.append((next_board, current_path + [move]))

    return -1, []  # No solution found

In [17]:
print("\nInitial State:")
for row in initial_state:
    print(row)
initial_board_tuple = tuple(tuple(row) for row in initial_state)
goal_board_tuple = tuple(tuple(row) for row in goal_state)

num_moves, path = dfs_solve_8_puzzle(initial_board_tuple, goal_board_tuple)

if num_moves != -1:
    print(f"\nSolution found! Number of moves: {num_moves}")
else:
    print("\nNo solution found for the given initial and goal states using DFS.")

print("\nGoal State:")
for row in goal_state:
    print(row)


Initial State:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

Solution found! Number of moves: 3437

Goal State:
[1, 2, 3]
[8, 0, 4]
[7, 6, 5]


In [19]:
# 8-Puzzle Solver using Iterative Deepening Depth-First Search (IDDFS)
import time

def get_blank_pos(state):
    """Finds the row and column index of the blank tile (0)."""
    idx = state.index(0)
    return idx // 3, idx % 3

def get_neighbors(state):
    """Generates valid adjacent states from the current state."""
    r, c = get_blank_pos(state)
    neighbors = []
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right

    for dr, dc in moves:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = list(state)
            idx1 = r * 3 + c
            idx2 = nr * 3 + nc
            new_state[idx1], new_state[idx2] = new_state[idx2], new_state[idx1]
            neighbors.append(tuple(new_state))
    return neighbors

def count_inversions(state):
    """Counts inversions in a state sequence (ignoring blank tile 0)."""
    arr = [tile for tile in state if tile != 0]
    inversions = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inversions += 1
    return inversions

def is_solvable(start, goal):
    """Checks if a start state can reach a custom goal state.
    Both must have the same inversion parity (both even or both odd)."""
    return (count_inversions(start) % 2) == (count_inversions(goal) % 2)

def depth_limited_dfs(state, goal, limit, path_len, visited):
    """Performs a depth-limited DFS tracking current branch to avoid cycles."""
    if state == goal:
        return path_len

    if limit <= 0:
        return None

    visited.add(state)

    for neighbor in get_neighbors(state):
        if neighbor not in visited:
            result = depth_limited_dfs(neighbor, goal, limit - 1, path_len + 1, visited)
            if result is not None:
                return result

    visited.remove(state) # Backtrack
    return None

def iddfs_solve(start, goal):
    """Main IDDFS wrapper that increments depth limit systematically."""
    depth = 0
    while True:
        visited = set()
        result_len = depth_limited_dfs(start, goal, depth, 0, visited)
        if result_len is not None:
            return result_len, depth
        depth += 1

def parse_input(prompt_text):
    """Validates and processes user input for a 3x3 layout state."""
    user_input = input(prompt_text).strip().split()
    if len(user_input) != 9:
        raise ValueError("You must enter exactly 9 space-separated digits.")

    state = tuple(int(x) for x in user_input)
    if sorted(state) != list(range(9)):
        raise ValueError("Input must contain unique digits from 0 to 8 inclusive.")
    return state

def main():


    try:
        # 1. Get Initial State
        start_state = parse_input("Enter initial state numbers: ")

        # 2. Get Goal State
        goal_state = parse_input("Enter target goal state numbers: ")

        # 3. Check compatibility
        if not is_solvable(start_state, goal_state):
            print("❌ This state transition is UNSOLVABLE. It is mathematically impossible to reach that specific goal configuration from this initial layout.")
            return

        print("\nSolving using Iterative Deepening DFS... (Please wait)")
        start_time = time.time()
        moves, depth = iddfs_solve(start_state, goal_state)
        end_time = time.time()

        print("\n--- SOLUTION FOUND ---")
        print(f"Total Number of Moves: {moves}")
        print(f"Final Depth Reached  : {depth}")
        print(f"Execution Time       : {end_time - start_time:.4f} seconds")

    except ValueError as e:
        print(f"Error: {e}")

if __name__ == '__main__':
    main()


Enter initial state numbers: 2 8 3 1 6 4 7 0 5
Enter target goal state numbers: 1 2 3 8 0 4 7 6 5

Solving using Iterative Deepening DFS... (Please wait)

--- SOLUTION FOUND ---
Total Number of Moves: 5
Final Depth Reached  : 5
Execution Time       : 0.0002 seconds
